In [1]:
import ee
import geemap

# 1. Xác thực và Khởi tạo Google Earth Engine
try:
    ee.Initialize()
    print("Khởi tạo GEE thành công!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
    print("Đã xác thực và khởi tạo GEE!")

# 2. Lấy ranh giới hành chính TP. Hồ Chí Minh (ROI) từ bộ dữ liệu FAO
# Lưu ý: Tên chuẩn trong database của FAO là 'Ho Chi Minh City'
hcm_roi = ee.FeatureCollection("FAO/GAUL/2015/level1") \
            .filter(ee.Filter.eq('ADM1_NAME', 'Ho Chi Minh City'))

# 3. Trực quan hóa thử ranh giới lên bản đồ
Map = geemap.Map()
Map.centerObject(hcm_roi, 10) # Zoom level 10
Map.addLayer(hcm_roi, {'color': 'red'}, 'TP. Ho Chi Minh ROI')
Map 


Successfully saved authorization token.
Đã xác thực và khởi tạo GEE!


c:\Users\Tan Nguyen\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1

  warnings.warn(warning, category=DeprecationWarning)


Map(center=[10.752267385049521, 106.69138504342378], controls=(WidgetControl(options=['position', 'transparent…

In [3]:
import ee
import geemap
import warnings

# Tắt các cảnh báo DeprecationWarning cho file code gọn gàng
warnings.filterwarnings('ignore')

# 1. Trở về bộ dữ liệu ranh giới chuẩn ban đầu (chắc chắn chạy được)
hcm_roi = ee.FeatureCollection("FAO/GAUL/2015/level1") \
            .filter(ee.Filter.eq('ADM1_NAME', 'Ho Chi Minh City'))

# 2. Hàm lọc mây (Cloud Masking) dựa trên dải QA_PIXEL của Landsat 8/9
def maskL89Clouds(image):
    qa = image.select('QA_PIXEL')
    cloudBitMask = 1 << 3
    shadowBitMask = 1 << 4
    mask = qa.bitwiseAnd(cloudBitMask).eq(0) \
             .And(qa.bitwiseAnd(shadowBitMask).eq(0))
    return image.updateMask(mask)

# 3. Hàm nội suy hệ số scale (Scale Factors) cho Landsat 8 Level 2
def apply_scale_factors(image):
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(opticalBands, None, True).addBands(thermalBands, None, True)

# 4. Truy vấn tập dữ liệu Landsat 8 (Giai đoạn mùa khô 2024 để ít mây nhất)
l8_collection = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
                 .filterBounds(hcm_roi)
                 .filterDate('2024-01-01', '2024-04-30') # Lọc ảnh 4 tháng đầu năm
                 .filter(ee.Filter.lt('CLOUD_COVER', 20)) # Lọc ảnh có độ phủ mây < 20%
                 .map(maskL89Clouds)       # Áp dụng hàm khử mây
                 .map(apply_scale_factors) # Áp dụng hàm scale
                 .median()                 # Lấy trung vị để ghép ra bức ảnh trong trẻo nhất
                 .clip(hcm_roi))           # Cắt đúng theo ranh giới TP.HCM

# 5. Trực quan hóa ảnh vệ tinh thực tế lên bản đồ
Map2 = geemap.Map()
Map2.centerObject(hcm_roi, 10)

# Cấu hình dải màu hiển thị (Băng 4-Đỏ, Băng 3-Xanh lục, Băng 2-Xanh lam)
vis_params = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 0.0,
    'max': 0.3
}

Map2.addLayer(l8_collection, vis_params, 'Landsat 8 - True Color 2024')
Map2.addLayer(hcm_roi, {'color': 'red', 'fillColor': '00000000'}, 'Ranh gioi TP.HCM')
Map2

Map(center=[10.752267385049615, 106.69138504342384], controls=(WidgetControl(options=['position', 'transparent…

In [4]:
# 1. Tính toán Chỉ số thực vật (NDVI)
ndvi = l8_collection.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')

# 2. Tính toán Nhiệt độ bề mặt (LST) sang độ C
# Hàm apply_scale_factors trước đó đã chuyển ST_B10 thành nhiệt độ Kelvin
lst = l8_collection.select('ST_B10').subtract(273.15).rename('LST')

# 3. Trực quan hóa LST và NDVI lên bản đồ
Map3 = geemap.Map()
Map3.centerObject(hcm_roi, 10)

# Cấu hình dải màu NDVI (Từ đỏ/vàng: đất trống/đô thị -> Xanh lục: thực vật dày)
ndvi_vis = {
    'min': -0.2,
    'max': 0.8,
    'palette': ['red', 'yellow', 'lightgreen', 'green', 'darkgreen']
}
Map3.addLayer(ndvi, ndvi_vis, 'Chỉ số NDVI')

# Cấu hình dải màu LST (Từ xanh dương/lục: Mát mẻ -> Vàng/Đỏ: Đảo nhiệt)
lst_vis = {
    'min': 25, 
    'max': 45, # Nhiệt độ giả định từ 25 đến 45 độ C
    'palette': ['blue', 'cyan', 'green', 'yellow', 'red', 'darkred']
}
Map3.addLayer(lst, lst_vis, 'Nhiệt độ LST (°C)')

# Khắc phục lỗi che khuất: Đục rỗng ranh giới, chỉ vẽ viền đỏ độ dày 2 pixel
empty_hcm = ee.Image().byte().paint(hcm_roi, 0, 2)
Map3.addLayer(empty_hcm, {'palette': ['red']}, 'Viền ranh giới TP.HCM')

Map3

Map(center=[10.752267385049615, 106.69138504342384], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
import pandas as pd

# 1. Gom các dải dữ liệu (Bands) cần thiết vào một bức ảnh (Image) duy nhất
training_image = ee.Image.cat([
    lst,
    ndvi,
    ee.Image.pixelLonLat()
])

# 2. Lấy mẫu ngẫu nhiên (Random Sampling) các pixel trong ranh giới TP.HCM
print("Đang trích xuất dữ liệu từ Google Earth Engine, vui lòng đợi khoảng 10-20 giây...")
sample_data = training_image.sample(
    region=hcm_roi.geometry(),
    scale=30,           
    numPixels=5000,     
    seed=42,            
    geometries=False    
)

# 3. Cách ép kiểu an toàn 100% (Không dùng geemap nữa)
# Kéo dữ liệu dạng JSON từ máy chủ Google về và chuyển sang Pandas DataFrame
features = sample_data.getInfo()['features']
data_list = [f['properties'] for f in features]
df = pd.DataFrame(data_list)

# 4. Tiền xử lý: Loại bỏ các dòng chứa giá trị rỗng (NaN) do bị cắt ở viền bản đồ
df = df.dropna()

# 5. Lưu lại thành file CSV để làm điểm checkpoint cho các bước sau
df.to_csv("HCMC_UHI_Dataset_2024.csv", index=False)

# Hiển thị 5 dòng đầu tiên để kiểm tra thành quả
print(f"Trích xuất thành công! Kích thước ma trận dữ liệu: {df.shape}")
display(df.head())

Đang trích xuất dữ liệu từ Google Earth Engine, vui lòng đợi khoảng 10-20 giây...
Trích xuất thành công! Kích thước ma trận dữ liệu: (4980, 4)


,LST,NDVI,latitude,longitude
0,31.378455,0.842087,10.476198,106.833717
1,41.065124,0.213465,10.898496,106.629440
2,37.976943,0.550038,11.013570,106.494424
3,30.501733,0.880452,10.511771,106.817548
4,32.388480,0.535056,10.410711,106.903516


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# 1. Tách biến đầu vào (X) và biến mục tiêu (y)
X = df[['NDVI', 'latitude', 'longitude']]
y = df['LST']

# 2. Chia tập dữ liệu: 80% để huấn luyện (Train), 20% để kiểm thử (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Số lượng mẫu huấn luyện: {X_train.shape[0]}")
print(f"Số lượng mẫu kiểm thử để đánh giá: {X_test.shape[0]}\n")

# 3. Khởi tạo mô hình
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)

# 4. Huấn luyện mô hình
print("Đang huấn luyện thuật toán Random Forest...")
rf_model.fit(X_train, y_train)

print("Đang huấn luyện thuật toán XGBoost...")
xgb_model.fit(X_train, y_train)

# 5. Hàm đánh giá mô hình dựa trên tiêu chí đề cương
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"--- Kết quả của {name} ---")
    print(f"RMSE: {rmse:.2f} °C \t(Tiêu chí: <= 2.0°C)")
    print(f"MAE:  {mae:.2f} °C \t(Tiêu chí: <= 1.5°C)")
    print(f"R²:   {r2:.2f} \t\t(Tiêu chí: >= 0.80)\n")
    return r2

# 6. Chạy đánh giá và so sánh
r2_rf = evaluate_model("Random Forest", rf_model, X_test, y_test)
r2_xgb = evaluate_model("XGBoost", xgb_model, X_test, y_test)

# 7. Xuất và lưu trữ mô hình tốt nhất để dùng cho WebGIS
best_model = rf_model if r2_rf > r2_xgb else xgb_model
model_name = "RandomForest" if r2_rf > r2_xgb else "XGBoost"

joblib.dump(best_model, 'best_uhi_model.joblib')
print(f"=> Đã lưu mô hình chiến thắng ({model_name}) vào file 'best_uhi_model.joblib'")

Số lượng mẫu huấn luyện: 3984
Số lượng mẫu kiểm thử để đánh giá: 996

Đang huấn luyện thuật toán Random Forest...
Đang huấn luyện thuật toán XGBoost...
--- Kết quả của Random Forest ---
RMSE: 2.57 °C 	(Tiêu chí: <= 2.0°C)
MAE:  1.52 °C 	(Tiêu chí: <= 1.5°C)
R²:   0.68 		(Tiêu chí: >= 0.80)

--- Kết quả của XGBoost ---
RMSE: 2.55 °C 	(Tiêu chí: <= 2.0°C)
MAE:  1.50 °C 	(Tiêu chí: <= 1.5°C)
R²:   0.68 		(Tiêu chí: >= 0.80)

=> Đã lưu mô hình chiến thắng (XGBoost) vào file 'best_uhi_model.joblib'


In [ ]:
import pandas as pd

# 1. Tính toán thêm Chỉ số mật độ xây dựng (NDBI)
ndbi = l8_collection.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')

# 2. Gom các dải dữ liệu (Thêm NDBI vào ma trận)
training_image_v2 = ee.Image.cat([
    lst,
    ndvi,
    ndbi,  # Biến số quan trọng quyết định nhiệt độ đô thị
    ee.Image.pixelLonLat()
])

# 3. Lấy mẫu ngẫu nhiên (Giới hạn tối đa 5000 điểm cho hàm getInfo)
print("Đang trích xuất bộ dữ liệu V2 (bổ sung NDBI), vui lòng đợi...")
sample_data_v2 = training_image_v2.sample(
    region=hcm_roi.geometry(),
    scale=30,           
    numPixels=5000,     # Hạ xuống 5000 để vượt qua giới hạn của Google
    seed=42,            
    geometries=False    
)

# 4. Kéo dữ liệu về Pandas DataFrame
features_v2 = sample_data_v2.getInfo()['features']
df_v2 = pd.DataFrame([f['properties'] for f in features_v2])

# Tiền xử lý: Bỏ giá trị rỗng
df_v2 = df_v2.dropna()

# 5. Lưu bộ dữ liệu mới và xem thử
df_v2.to_csv("HCMC_UHI_Dataset_2024_V2.csv", index=False)
print(f"Trích xuất thành công V2! Kích thước ma trận dữ liệu: {df_v2.shape}")
display(df_v2.head())

Đang trích xuất bộ dữ liệu V2 (bổ sung NDBI), vui lòng đợi...
Trích xuất thành công V2! Kích thước ma trận dữ liệu: (4978, 5)


,LST,NDBI,NDVI,latitude,longitude
0,31.378455,-0.585540,0.842087,10.476198,106.833717
1,41.065124,0.066893,0.213465,10.898496,106.629440
2,37.976943,-0.068366,0.550038,11.013570,106.494424
3,30.501733,-0.622916,0.880452,10.511771,106.817548
4,32.388480,-0.386235,0.535056,10.410711,106.903516


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# 1. Tách biến đầu vào (X) có thêm NDBI và biến mục tiêu (y)
X = df_v2[['NDVI', 'NDBI', 'latitude', 'longitude']]
y = df_v2['LST']

# 2. Chia tập dữ liệu: 80% để huấn luyện (Train), 20% để kiểm thử (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Số lượng mẫu huấn luyện: {X_train.shape[0]}")
print(f"Số lượng mẫu kiểm thử để đánh giá: {X_test.shape[0]}\n")

# 3. Khởi tạo mô hình
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)

# 4. Huấn luyện mô hình
print("Đang huấn luyện thuật toán Random Forest...")
rf_model.fit(X_train, y_train)

print("Đang huấn luyện thuật toán XGBoost...")
xgb_model.fit(X_train, y_train)

# 5. Hàm đánh giá mô hình
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"--- Kết quả của {name} ---")
    print(f"RMSE: {rmse:.2f} °C \t(Tiêu chí: <= 2.0°C)")
    print(f"MAE:  {mae:.2f} °C \t(Tiêu chí: <= 1.5°C)")
    print(f"R²:   {r2:.2f} \t\t(Tiêu chí: >= 0.80)\n")
    return r2

# 6. Chạy đánh giá và so sánh
r2_rf = evaluate_model("Random Forest", rf_model, X_test, y_test)
r2_xgb = evaluate_model("XGBoost", xgb_model, X_test, y_test)

# 7. Xuất và lưu trữ mô hình tốt nhất
best_model = rf_model if r2_rf > r2_xgb else xgb_model
model_name = "RandomForest" if r2_rf > r2_xgb else "XGBoost"

joblib.dump(best_model, 'best_uhi_model_v2.joblib')
print(f"=> Đã lưu mô hình chiến thắng ({model_name}) vào file 'best_uhi_model_v2.joblib'")

Số lượng mẫu huấn luyện: 3982
Số lượng mẫu kiểm thử để đánh giá: 996

Đang huấn luyện thuật toán Random Forest...
Đang huấn luyện thuật toán XGBoost...
--- Kết quả của Random Forest ---
RMSE: 1.93 °C 	(Tiêu chí: <= 2.0°C)
MAE:  1.33 °C 	(Tiêu chí: <= 1.5°C)
R²:   0.79 		(Tiêu chí: >= 0.80)

--- Kết quả của XGBoost ---
RMSE: 1.93 °C 	(Tiêu chí: <= 2.0°C)
MAE:  1.31 °C 	(Tiêu chí: <= 1.5°C)
R²:   0.79 		(Tiêu chí: >= 0.80)

=> Đã lưu mô hình chiến thắng (XGBoost) vào file 'best_uhi_model_v2.joblib'


In [ ]:
from sklearn.model_selection import GridSearchCV

print("Đang chạy Grid Search để rà soát siêu tham số tối ưu cho XGBoost...")
print("Quá trình này có thể mất 1-2 phút, vui lòng đợi...\n")

# 1. Khai báo lưới tham số cần thử nghiệm (Tăng số lượng cây, chỉnh độ sâu và tốc độ học)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 7, 9],
    'learning_rate': [0.05, 0.1, 0.15],
    'subsample': [0.8, 1.0]
}

# 2. Khởi tạo mô hình và Grid Search
xgb_base = XGBRegressor(random_state=42)
grid_search = GridSearchCV(
    estimator=xgb_base, 
    param_grid=param_grid, 
    cv=3,             # Chia tập Train làm 3 phần để kiểm định chéo (Cross-validation)
    scoring='r2',     # Mục tiêu tối thượng là tối đa hóa R2
    n_jobs=-1,        # Dùng toàn bộ lõi CPU để chạy cho nhanh
    verbose=1
)

# 3. Tiến hành đào tạo và tìm kiếm
grid_search.fit(X_train, y_train)

# 4. Trích xuất mô hình xịn nhất
best_xgb_tuned = grid_search.best_estimator_
print("\n=> Đã tìm thấy bộ tham số tối ưu:", grid_search.best_params_)
print("-" * 40)

# 5. Đánh giá lại với mô hình đã độ tham số
r2_final = evaluate_model("XGBoost (Đã tinh chỉnh)", best_xgb_tuned, X_test, y_test)

# 6. Lưu file chính thức
if r2_final >= 0.80:
    joblib.dump(best_xgb_tuned, 'final_uhi_model.joblib')
    print("=> Chúc mừng! Đã lưu mô hình hoàn chỉnh vào 'final_uhi_model.joblib'")

Đang chạy Grid Search để rà soát siêu tham số tối ưu cho XGBoost...
Quá trình này có thể mất 1-2 phút, vui lòng đợi...

Fitting 3 folds for each of 54 candidates, totalling 162 fits

=> Đã tìm thấy bộ tham số tối ưu: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
----------------------------------------
--- Kết quả của XGBoost (Đã tinh chỉnh) ---
RMSE: 1.91 °C 	(Tiêu chí: <= 2.0°C)
MAE:  1.30 °C 	(Tiêu chí: <= 1.5°C)
R²:   0.80 		(Tiêu chí: >= 0.80)



In [ ]:
import ee
import pandas as pd

# Khởi tạo danh sách rỗng để hứng dữ liệu từng năm
all_dataframes = []

print("Bắt đầu tải dữ liệu trực tiếp về máy tính (không cần qua Google Drive)...")

# Vòng lặp quét qua 10 năm (từ 2017 đến hết 2026)
for year in range(2017, 2027):
    print(f"Đang xử lý và kéo dữ liệu năm {year}...")
    
    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = ee.Date.fromYMD(year, 12, 31)
    
    # Lọc và đục mây ảnh Landsat theo năm
    yearly_col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
                 .filterBounds(hcm_roi)
                 .filterDate(start_date, end_date)
                 .filter(ee.Filter.lt('CLOUD_COVER', 20))
                 .map(maskL89Clouds)
                 .map(apply_scale_factors)
                 .median()
                 .clip(hcm_roi))
    
    # Tính LST, NDVI, NDBI
    ndvi = yearly_col.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    ndbi = yearly_col.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')
    lst = yearly_col.select('ST_B10').subtract(273.15).rename('LST')
    
    # Gắn nhãn Năm (Year) vào dữ liệu
    year_img = ee.Image.constant(year).rename('Year').toInt()
    
    combined = ee.Image.cat([lst, ndvi, ndbi, year_img, ee.Image.pixelLonLat()])
    
    # Lấy 2000 điểm (Vượt qua an toàn mức chặn 5000 của getInfo)
    samples = combined.sample(
        region=hcm_roi.geometry(),
        scale=30,
        numPixels=2000,
        seed=42,
        geometries=False
    )
    
    # Ép kiểu trực tiếp về Pandas DataFrame cục bộ
    try:
        features = samples.getInfo()['features']
        df_year = pd.DataFrame([f['properties'] for f in features])
        all_dataframes.append(df_year)
    except Exception as e:
        print(f"Lỗi nhẹ ở năm {year}, bỏ qua và tiếp tục: {e}")

# Gom toàn bộ dữ liệu 10 năm lại thành một bảng duy nhất
final_df = pd.concat(all_dataframes, ignore_index=True)

# Dọn dẹp dữ liệu rỗng (NaN)
final_df = final_df.dropna()

# Lưu thẳng vào ổ cứng máy tính (cùng thư mục với file code)
final_df.to_csv("HCMC_UHI_10Years_2017_2026.csv", index=False)

print(f"\n=> Xong! Đã lưu file 'HCMC_UHI_10Years_2017_2026.csv' trực tiếp vào máy tính.")
print(f"=> Tổng kích thước siêu ma trận: {final_df.shape}")
display(final_df.head(3))
display(final_df.tail(3))

Bắt đầu tải dữ liệu trực tiếp về máy tính (không cần qua Google Drive)...
Đang xử lý và kéo dữ liệu năm 2017...
Đang xử lý và kéo dữ liệu năm 2018...
Đang xử lý và kéo dữ liệu năm 2019...
Đang xử lý và kéo dữ liệu năm 2020...
Đang xử lý và kéo dữ liệu năm 2021...
Đang xử lý và kéo dữ liệu năm 2022...
Đang xử lý và kéo dữ liệu năm 2023...
Đang xử lý và kéo dữ liệu năm 2024...
Đang xử lý và kéo dữ liệu năm 2025...
Đang xử lý và kéo dữ liệu năm 2026...

=> Xong! Đã lưu file 'HCMC_UHI_10Years_2017_2026.csv' trực tiếp vào máy tính.
=> Tổng kích thước siêu ma trận: (19937, 6)


,LST,NDBI,NDVI,Year,latitude,longitude
0,32.784970,-0.580002,0.840443,2017,10.476198,106.833717
1,40.111496,-0.000094,0.247263,2017,10.898496,106.629440
2,33.639475,-0.178130,0.661504,2017,11.013570,106.494424


,LST,NDBI,NDVI,Year,latitude,longitude
19934,30.354758,-0.675336,0.873532,2026,10.447901,106.849617
19935,38.628075,0.037392,0.469981,2026,10.772103,106.489842
19936,44.382312,0.081525,0.114782,2026,10.868851,106.763110


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# 1. Khai báo biến đầu vào (X) - Chú ý ĐÃ CÓ THÊM CỘT 'Year'
X = final_df[['NDVI', 'NDBI', 'Year', 'latitude', 'longitude']]
y = final_df['LST']

# 2. Chia tập dữ liệu: 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Kích thước tập huấn luyện: {X_train.shape[0]} mẫu")
print(f"Kích thước tập kiểm thử để đánh giá: {X_test.shape[0]} mẫu\n")

# 3. Khởi tạo thuật toán XGBoost với bộ tham số xịn nhất đã tìm được
xgb_ultimate = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

# 4. Huấn luyện mô hình
print("Đang huấn luyện mô hình XGBoost với dữ liệu lịch sử 10 năm...")
xgb_ultimate.fit(X_train, y_train)

# 5. Đánh giá độ chính xác
y_pred = xgb_ultimate.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- KẾT QUẢ NGHIỆM THU CUỐI CÙNG ---")
print(f"RMSE: {rmse:.2f} °C \t(Tiêu chí: <= 2.0°C)")
print(f"MAE:  {mae:.2f} °C \t(Tiêu chí: <= 1.5°C)")
print(f"R²:   {r2:.2f} \t\t(Tiêu chí: >= 0.80)\n")

# 6. Lưu file model chính thức
joblib.dump(xgb_ultimate, 'ultimate_uhi_model_10yrs.joblib')
print("=> XONG! File 'ultimate_uhi_model_10yrs.joblib' đã sẵn sàng.")
print("=> Bây giờ, chỉ cần truyền vào tham số Year=2030, mô hình có thể dự báo được tương lai!")

Kích thước tập huấn luyện: 15949 mẫu
Kích thước tập kiểm thử để đánh giá: 3988 mẫu

Đang huấn luyện mô hình XGBoost với dữ liệu lịch sử 10 năm...
--- KẾT QUẢ NGHIỆM THU CUỐI CÙNG ---
RMSE: 1.93 °C 	(Tiêu chí: <= 2.0°C)
MAE:  1.41 °C 	(Tiêu chí: <= 1.5°C)
R²:   0.84 		(Tiêu chí: >= 0.80)

=> XONG! File 'ultimate_uhi_model_10yrs.joblib' đã sẵn sàng.
=> Bây giờ, chỉ cần truyền vào tham số Year=2030, mô hình có thể dự báo được tương lai!
